# ภารกิจเสริม - ชุดต้นไม้ (Trees Ensemble)

ในโน้ตบุ๊กนี้ คุณจะ:
 - ใช้ Pandas เพื่อดำเนินการเข้ารหัสแบบ one-hot ของชุดข้อมูล
 - ใช้ scikit-learn เพื่อนำไปใช้กับโมเดล Decision Tree, Random Forest และ XGBoost

มานำเข้าไลบรารีที่คุณจะใช้กันเถอะ

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
try:
  %matplotlib widget
  print("widget is already installed")
except:
  print("widget is not been installed, install now..")
  !pip install ipympl

In [ ]:
!git clone https://github.com/Smith-WeStrideTH/Advance_Learning_Algorithm_Course.git
%cd Advance_Learning_Algorithm_Course/work 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
!pip install xgboost --quiet
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

RANDOM_STATE = 55 ## You will pass it to every sklearn call so we ensure reproducibility

# 1. โหลดชุดข้อมูล

แหล่งที่มาของข้อมูล (Source): [Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)

บริบท (Context)
โรคหัวใจและหลอดเลือด (CVDs) เป็นสาเหตุอันดับ 1 ของการเสียชีวิตทั่วโลก โดยคร่าชีวิตผู้คนไปประมาณ 17.9 ล้านคนต่อปี คิดเป็นร้อยละ 31 ของการเสียชีวิตทั้งหมดทั่วโลก ภาวะหัวใจล้มเหลวเป็นภาวะการณ์ทั่วไปที่เกิดจากโรคหัวใจและหลอดเลือด (CVDs) ชุดข้อมูลนี้มีคุณลักษณะ 11 ประการที่สามารถใช้ทำนายโรคหัวใจที่อาจเกิดขึ้นได้

ผู้ป่วยโรคหัวใจและหลอดเลือดหรือผู้ที่มีความเสี่ยงสูงต่อโรคหัวใจและหลอดเลือด จำเป็นต้องได้รับการตรวจวินิจฉัยและการจัดการตั้งแต่เนิ่นๆ ซึ่งโมเดลการเรียนรู้ของเครื่อง (Machine Learning Model) สามารถช่วยเหลือได้อย่างมาก

คุณจะพัฒนาโมเดลเพื่อทำนายโอกาสที่บุคคลใดบุคคลหนึ่งจะเกิดโรคหัวใจและหลอดเลือด โดยอาศัยข้อมูลทั้งหมดด้านล่างนี้

#### ข้อมูลแอตทริบิวต์ (Attribute Information)

- อายุ (Age): อายุของผู้ป่วย [ปี] 
- เพศ (Sex): เพศของผู้ป่วย [ชาย: ชาย, หญิง: หญิง]
- ประเภทอาการเจ็บหน้าอก (ChestPainType): ประเภทอาการเจ็บหน้าอก [TA: อาการเจ็บหน้าอกแบบปกติ, ATA: อาการเจ็บหน้าอกแบบผิดปกติ, NAP: อาการเจ็บไม่ร้าวไปยังบริเวณอื่น, ASY: ไม่มีอาการ]
- ความดันโลหิตขณะพัก (RestingBP): ความดันโลหิตขณะพัก [มม.ปรอท]
- คอเลสเตอรอล (Cholesterol): คอเลสเตอรอลในเลือด [มก./ดล.]
- น้ำตาลในเลือดขณะท้องว่าง (FastingBS): น้ำตาลในเลือดขณะท้องว่าง [1: หากน้ำตาลในเลือดขณะท้องว่าง > 120 มก./ดล., 0: มิเช่นนั้น]
- ผลการตรวจคลื่นไฟฟ้าหัวใจขณะพัก (RestingECG): ผลการตรวจคลื่นไฟฟ้าหัวใจขณะพัก [ปกติ: ปกติ, ST: มีความผิดปกติของคลื่น ST-T (คลื่น T หัวกลับและ/หรือ ST สูงขึ้นหรือต่ำลงมากกว่า 0.05 mV), LVH: แสดงภาวะกล้ามเนื้อหัวใจห้องล่างหนาแน่น โดยเกณฑ์ของ Estes]
- อัตราการเต้นของหัวใจสูงสุด (MaxHR): อัตราการเต้นของหัวใจสูงสุดที่วัดได้ [ค่าตัวเลขระหว่าง 60 ถึง 202]
- อาการเจ็บหน้าอกขณะออกกำลังกาย (ExerciseAngina): อาการเจ็บหน้าอกขณะออกกำลังกาย [Y: ใช่, N: ไม่ใช่]
- Oldpeak: oldpeak = ST [ค่าตัวเลขที่วัดเป็นความลึก]
- ST_Slope: ความลาดชันของส่วนยอดของคลื่น ST ช่วงออกกำลังกาย [Up: ลาดขึ้น, Flat: ราบ, Down: ลาดลง]
- โรคหัวใจ (HeartDisease): คลาสผลลัพธ์ [1: โรคหัวใจ, 0: ปกติ]

ตอนนี้มาโหลดชุดข้อมูลกัน ขณะที่คุณเห็นด้านบน ตัวแปรเหล่านี้:
- เพศ (Sex)

- ประเภทอาการเจ็บหน้าอก (ChestPainType)
- การตรวจคลื่นไฟฟ้าหัวใจขณะพัก (RestingECG)

- อาการเจ็บหน้าอกขณะออกกำลังกาย (ExerciseAngina)

- ความชันของ ST (ST_Slope)


เป็น ประเภทหมวดหมู่  *categorical*,  ดังนั้นคุณต้องเข้ารหัสแบบ one-hot

In [ ]:
# Load the dataset using pandas
df = pd.read_csv("heart.csv")

In [ ]:
df.head()

คุณต้องดำเนินการวิศวกรรมข้อมูลก่อนทำงานกับโมเดล มีคุณลักษณะเชิงหมวดหมู่ 5 รายการ ดังนั้นคุณจะใช้ Pandas เพื่อเข้ารหัสแบบ one-hot

## 2. การเข้ารหัสแบบหนึ่งร้อนโดยใช้ Pandas

ขั้นแรก คุณจะต้องลบคอลัมน์ไบนารีออก เนื่องจากการเข้ารหัสแบบหนึ่งร้อนกับคอลัมน์ไบนารีจะไม่มีผลอะไรต่อคอลัมน์เหล่านั้น เพื่อให้บรรลุเป้าหมายนี้ คุณจะนับจำนวนค่าที่แตกต่างกันในแต่ละตัวแปรเชิงหมวดหมู่และพิจารณาเฉพาะตัวแปรที่มีค่ามากกว่า 3 ค่าเท่านั้น

In [ ]:
cat_variables = ['Sex',
'ChestPainType',
'RestingECG',
'ExerciseAngina',
'ST_Slope'
]

เพื่อเตือนความจำ การเข้ารหัสแบบ one-hot มีจุดมุ่งหมายเพื่อแปลงตัวแปรเชิงหมวดหมู่ที่มี `n` ผลลัพธ์ให้เป็น `n` ตัวแปรแบบไบนารี

Pandas มีวิธีการในตัวเพื่อเข้ารหัสแบบ one-hot ตัวแปร นั่นคือฟังก์ชัน`pd.get_dummies`.ฟังก์ชันนี้มีอาร์กิวเมนต์หลายตัว แต่คุณจะใช้เพียงไม่กี่ตัวเท่านั้น นั่นคือ:

 - data: DataFrame ที่จะใช้

 - prefix: รายการที่มีคำนำหน้า เพื่อให้คุณทราบว่าคุณกำลังจัดการกับค่าใด

 - columns: รายการของคอลัมน์ที่จะถูกเข้ารหัสแบบ one-hot 'prefix' และ 'columns' ต้องมีความยาวเท่ากัน

 
สำหรับข้อมูลเพิ่มเติม คุณสามารถพิมพ์ `help(pd.get_dummies)`  เพื่ออ่านเอกสารประกอบฉบับเต็มของฟังก์ชันได้เสมอ

In [ ]:
# This will replace the columns with the one-hot encoded ones and keep the columns outside 'columns' argument as it is.
df = pd.get_dummies(data = df,
                         prefix = cat_variables,
                         columns = cat_variables)

In [ ]:
df.head()

ตอนนี้คุณจะกำหนดชุดตัวแปรสุดท้ายที่จะถูกใช้โดยโมเดลที่คุณจะสร้างในห้องปฏิบัติการนี้

In [ ]:
var = [x for x in df.columns if x not in 'HeartDisease'] ## Removing our target variable

สังเกตว่าจำนวนตัวแปรได้เปลี่ยนไป คุณเริ่มต้นด้วย 11 ตัวแปร ตอนนี้คุณมี:

In [ ]:
print(len(var))

# 3. การแบ่งชุดข้อมูล

ในส่วนนี้ คุณจะแบ่งชุดข้อมูลของคุณออกเป็นชุดฝึกและชุดทดสอบ คุณจะใช้ฟังก์ชัน  `train_test_split`จาก Scikit-learn มาดูอาร์กิวเมนต์ของมันกัน

In [ ]:
help(train_test_split)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df[var], df['HeartDisease'], train_size = 0.8, random_state = RANDOM_STATE)

# We will keep the shuffle = True since our dataset has not any time dependency.

In [ ]:
print(f'train samples: {len(X_train)}\ntest samples: {len(X_test)}')
print(f'target proportion: {sum(y_train)/len(y_train):.4f}')

# 4. สร้างโมเดล

## 4.1 ต้นไม้ตัดสินใจ (Decision Tree)


ในส่วนนี้ เราจะมาทำงานกับต้นไม้ตัดสินใจที่คุณเรียนรู้ไปก่อนหน้านี้ แต่ตอนนี้จะใช้การนำไปประยุกต์ใช้กับ  [Scikit-learn implementation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). 

Object ต้นไม้ตัดสินใจใน Scikit-learn มีไฮเปอร์พารามิเตอร์ (Hyperparameter) หลายตัว ในแลปนี้ คุณจะใช้เพียงบางส่วนเท่านั้น และคุณจะไม่ดำเนินการเลือกฟีเจอร์ (Feature Selection) หรือปรับแต่งไฮเปอร์พารามิเตอร์ (แต่แนะนำให้ลองทำเพื่อเปรียบเทียบผลลัพธ์ดู!) :-) )


ไฮเปอร์พารามิเตอร์ ที่คุณจะใช้และศึกษามีดังนี้:

 - min_samples_split: จำนวนตัวอย่างขั้นต่ำที่จำเป็นสำหรับการแยกโหนดภายใน ซึ่งอาจช่วยป้องกันการโอเวอร์ฟิตติ้ง (Overfitting)
 - max_depth: ความลึกสูงสุดของต้นไม้ ซึ่งอาจช่วยป้องกันการโอเวอร์ฟิตติ้ง (Overfitting)

In [ ]:
min_samples_split_list = [2,10, 30, 50, 100, 200, 300, 700] ## If the number is an integer, then it is the actual quantity of samples,
max_depth_list = [1,2, 3, 4, 8, 16, 32, 64, None] # None means that there is no depth limit.

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for min_samples_split in min_samples_split_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = DecisionTreeClassifier(min_samples_split = min_samples_split,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('min_samples_split')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(min_samples_split_list )),labels=min_samples_split_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

สังเกตว่าการเพิ่มจำนวน `min_samples_split` จะลดการโอเวอร์ฟิต

มาทำการทดลองเดียวกันกับ `max_depth`.

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for max_depth in max_depth_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = DecisionTreeClassifier(max_depth = max_depth,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(max_depth_list )),labels=max_depth_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

ความแม่นยำของการทดสอบสูงสุดที่ tree_depth=3 เมื่อความลึกที่อนุญาตมีขนาดเล็ก ต้นไม้ไม่สามารถสร้างการแยกแยะได้เพียงพอเพื่อแยกแยะบวกจากลบ (มีปัญหา underfit) แต่เมื่อความลึกที่อนุญาตสูงเกินไป (>= 5) ต้นไม้จะเชี่ยวชาญเกินไปกับชุดฝึกและทำให้สูญเสียความแม่นยำไปยังชุดทดสอบ (มีปัญหา overfit) โมเดลต้นไม้สุดท้ายของเราจะมี:

- `max_depth = 3`
- `min_samples_split = 50` 

In [ ]:
decision_tree_model = DecisionTreeClassifier(min_samples_split = 50,
                                             max_depth = 3,
                                             random_state = RANDOM_STATE).fit(X_train,y_train)

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_test),y_test):.4f}")

ไม่มีสัญญาณของการโอเวอร์ฟิต แม้ว่าเมตริกจะไม่ดีนักก็ตาม

## 4.2 Random Forest

ตอนนี้ลองใช้อัลกอริทึม Random Forest ด้วยการใช้การใช้งาน Scikit-learn โดยธรรมชาติ ไฮเปอร์พารามิเตอร์ทั้งหมดข้างต้นจะมีอยู่ในอัลกอริทึมนี้ เนื่องจากเป็นเพียงชุดของต้นตัดสินใจ แต่จะมีไฮเปอร์พารามิเตอร์อื่นที่คุณจะใช้ เรียกว่า `n_estimators` ซึ่งเป็นจำนวนต้นตัดสินใจที่จะมีการปรับแต่ง

โปรดจำไว้ว่าสำหรับ Random Forest คุณจะใช้ชุดย่อยของคุณสมบัติและชุดย่อยของชุดฝึกเพื่อฝึกต้นไม้แต่ละต้น ซึ่งเลือกแบบสุ่ม ในกรณีนี้ คุณจะใช้จำนวนคุณสมบัติตามที่คุณเห็นในบรรยาย ซึ่งเป็น $\sqrt{n}$
โดยที่ $n$ คือจำนวนคุณสมบัติ อย่างไรก็ตาม สามารถปรับเปลี่ยนได้ สำหรับข้อมูลเพิ่มเติมเกี่ยวกับไฮเปอร์พารามิเตอร์ของ Random Forest คุณสามารถเรียกใช้  `help(RandomForestClassifier)`.

อีกพารามิเตอร์หนึ่งที่ไม่มีผลกระทบต่อผลลัพธ์สุดท้าย แต่สามารถเร่งการคำนวณได้ เรียกว่า  `n_jobs` เนื่องจากการปรับแต่งของแต่ละต้นไม้เป็นอิสระจากกัน จึงสามารถเรียกใช้การปรับแต่งแบบขนานได้ ดังนั้นการตั้งค่า `n_jobs`  ที่สูงขึ้นจะเพิ่มจำนวนคอร์ CPU ที่จะใช้ โปรดทราบว่าตัวเลขที่ใกล้เคียงกับจำนวนคอร์สูงสุดของ CPU ของคุณมากอาจส่งผลต่อประสิทธิภาพโดยรวมของพีซีของคุณและอาจนำไปสู่การหยุดทำงาน

คุณจะเรียกใช้สคริปต์เดียวกันอีกครั้ง แต่มีพารามิเตอร์อื่นคือ `n_estimators`ซึ่งเราจะเลือกระหว่าง 10, 50 และ 100 ค่าเริ่มต้นคือ 100

In [ ]:
min_samples_split_list = [2,10, 30, 50, 100, 200, 300, 700]  ## If the number is an integer, then it is the actual quantity of samples,
                                             ## If it is a float, then it is the percentage of the dataset
max_depth_list = [2, 4, 8, 16, 32, 64, None]
n_estimators_list = [10,50,100,500]

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for min_samples_split in min_samples_split_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(min_samples_split = min_samples_split,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('min_samples_split')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(min_samples_split_list )),labels=min_samples_split_list) 
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for max_depth in max_depth_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(max_depth = max_depth,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(max_depth_list )),labels=max_depth_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for n_estimators in n_estimators_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(n_estimators = n_estimators,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('n_estimators')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(n_estimators_list )),labels=n_estimators_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

มาดูการปรับฟิตของแบบจำลองป่าสุ่มด้วยพารามิเตอร์ต่อไปนี้กัน:

- max_depth: 8
- min_samples_split: 10
- n_estimators: 100

In [ ]:
random_forest_model = RandomForestClassifier(n_estimators = 100,
                                             max_depth = 8, 
                                             min_samples_split = 10).fit(X_train,y_train)

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_test),y_test):.4f}")

คุณได้แสดงให้เห็นถึงวิธีการค้นหาค่าไฮเปอร์พารามิเตอร์ที่ดีที่สุดโดยไฮเปอร์พารามิเตอร์ อย่างไรก็ตาม คุณไม่ควรมองข้ามว่าเมื่อเราทดลองกับไฮเปอร์พารามิเตอร์หนึ่ง เราต้องแก้ไขค่าเริ่มต้นของค่าอื่น ๆ เสมอ สิ่งนี้ทำให้เราสามารถบอกได้ว่าค่าไฮเปอร์พารามิเตอร์เปลี่ยนแปลงไปตามค่าเริ่มต้นเหล่านั้นเท่านั้น

ในทางทฤษฎี หากคุณมีค่า 4 ค่าที่จะลองใช้ในแต่ละไฮเปอร์พารามิเตอร์ 3 ตัวที่กำลังปรับแต่ง คุณควรจะมีทั้งหมด 4 x 4 x 4 = 64 ชุดค่าผสม อย่างไรก็ตาม วิธีที่คุณกำลังทำจะให้ผลลัพธ์เพียง 4 + 4 + 4 = 12 ชุดค่าผสมเท่านั้น เพื่อลองใช้ชุดค่าผสมทั้งหมด คุณสามารถใช้การนำไปใช้ GridSearchCV ของ sklearn นอกจากนี้ ยังมีพารามิเตอร์ refit ที่จะปรับโมเดลโดยอัตโนมัติบนชุดค่าผสมที่ดีที่สุด ดังนั้นคุณจะไม่จำเป็นต้องเขียนโปรแกรมอย่างชัดเจน สำหรับข้อมูลเพิ่มเติมเกี่ยวกับ GridSearchCV โปรดดูที่ [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

## 4.3 XGBoost

ตอนนี้ โมเดลสุดท้ายที่คุณจะทดสอบในห้องปฏิบัติการนี้คือโมเดล Gradient Boosting ที่เรียกว่า XGBoost ตามที่คุณได้เห็นในบทเรียนแล้ว วิธีการบูสต์จะฝึกต้นไม้หลายต้น แต่แทนที่จะไม่สัมพันธ์กัน ต้นไม้จะถูกปรับให้เหมาะสมตามลำดับเพื่อลดข้อผิดพลาด

พารามิเตอร์ที่โมเดลนี้ประกอบด้วยเหมือนกับพารามิเตอร์สำหรับต้นไม้การตัดสินใจใด ๆ รวมถึงพารามิเตอร์อื่น ๆ เช่น อัตราการเรียนรู้ ซึ่งเป็นขนาดของขั้นตอนในการใช้วิธีการ Gradient Descent ที่ XGBoost ใช้ภายในเพื่อลดข้อผิดพลาดในแต่ละขั้นตอนการฝึก

สิ่งที่น่าสนใจอย่างหนึ่งเกี่ยวกับ XGBoost คืออนุญาตให้ระหว่างการปรับให้เหมาะสมสามารถส่งผ่านชุดข้อมูลการประเมินรายการของรูปแบบ `(X_val,y_val)`ซึ่งในแต่ละรอบจะวัดค่าใช้จ่าย (หรือเมตริกการประเมิน) ในชุดข้อมูลการประเมิน เพื่อให้เมื่อค่าใช้จ่าย (หรือเมตริก) หยุดลดลงเป็นจำนวนรอบ (เรียกว่า early_stopping_rounds) การฝึกจะหยุด นี่คือวิธีที่เราสามารถควบคุมจำนวนตัวประมาณค่าได้โดยอัตโนมัติ และวิธีที่เราสามารถหลีกเลี่ยงการโอเวอร์ฟิตติ้งเนื่องจากตัวประมาณค่ามากเกินไป

อันดับแรก มาสร้างชุดย่อยของชุดฝึกของเรา (เราไม่ควรใช้ชุดทดสอบที่นี่)



In [ ]:
n = int(len(X_train)*0.8) ## Let's use 80% to train and 20% to eval

In [ ]:
X_train_fit, X_train_eval, y_train_fit, y_train_eval = X_train[:n], X_train[n:], y_train[:n], y_train[n:]

คุณสามารถตั้งค่าตัวประมาณการจำนวนมากได้ เนื่องจากคุณสามารถหยุดได้หากฟังก์ชันต้นทุนหยุดลดลง

In [ ]:
xgb_model = XGBClassifier(n_estimators = 500, learning_rate = 0.1,verbosity = 1, random_state = RANDOM_STATE, early_stopping_rounds = 50)
xgb_model.fit(X_train_fit,y_train_fit, 
              eval_set = [(X_train_eval,y_train_eval)])
# Here we must pass a list to the eval_set, because you can have several different tuples ov eval sets. The parameter 
# early_stopping_rounds is the number of iterations that it will wait to check if the cost function decreased or not.
# If not, it will stop and get the iteration that returned the lowest metric on the eval set.

อย่างที่คุณเห็น แม้ว่าคุณจะส่งผ่านตัวประมาณค่า 500 ตัวเพื่อให้พอดี แต่อัลกอริทึมก็พอดีเพียง 66 ตัวเท่านั้น เนื่องจากการสูญเสียแบบล็อกที่ใช้ในการวัดรอบการฝึกเริ่มเพิ่มขึ้น ในความเป็นจริง จำนวนตัวประมาณค่านั้นน้อยกว่า 66 ตัว ถ้าคุณมองเมตริกอย่างใกล้ชิด คุณจะเห็นว่าด้วยต้นไม้ที่พอดี 16 ต้น เราได้ค่าต่ำสุดของการสูญเสียแบบล็อก และในความเป็นจริง นี่คือจำนวนต้นไม้ที่พอดีในโมเดลสุดท้าย

In [ ]:
xgb_model.best_iteration

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_test),y_test):.4f}")

คุณสามารถเห็นได้ว่า RandomForest บรรลุความแม่นยำที่ดีที่สุด แต่ผลลัพธ์โดยรวมนั้นใกล้เคียงกัน และโปรดทราบว่าเราได้เมตริกการทดสอบที่ใกล้เคียงมากกับ XGBoost เมื่อเทียบกับ RandomForest และเราไม่ได้ทำการค้นหาไฮเปอร์พารามิเตอร์เลย! ข้อดีของ XGBoost คือเร็วกว่า Random Forest และยังมีพารามิเตอร์มากกว่า ดังนั้นคุณจึงสามารถปรับแต่งโมเดลเพื่อให้ได้ผลลัพธ์ที่ดีขึ้น

ขอแสดงความยินดี คุณได้เรียนรู้วิธีการใช้ Decision Tree, Random Forest จากไลบรารี scikit-learn และ XGBoost แล้ว!